# CMIP6 (NEX-GDDP-CMIP6) Precipitation Extraction via Google Earth Engine

Extracts daily precipitation for the **representative rainfall stations** (from the
Tab 1 app/notebook) from the **NASA/GDDP-CMIP6** dataset for one or more GCMs,
converts units to mm/day, and saves each model's output in the **same Date x Station
format** as `Representative_Stations_Rainfall.xlsx` — so it can be directly compared
with IMD observed rainfall (R2, NSE, RMSE, PBIAS, KGE, ranking).

**Steps**
1. Authenticate Earth Engine
2. Upload `Final_Stations.xlsx` (from Tab 1) — gives Station_ID + Latitude + Longitude
3. Pick GCM model(s) and scenario + year range
4. Run extraction (chunked by years to keep requests small)
5. Download one Excel file per model (Date | ST001 | ST002 | ...)

**Note on resolution**: NEX-GDDP-CMIP6 is ~25km (0.25°) resolution, comparable to IMD,
but a single GCM grid cell can still cover more than one nearby station — that's
expected and fine for comparison purposes.


## 1. Authenticate Earth Engine

In [ ]:
import ee

# First time: this opens a browser auth flow.
ee.Authenticate()

# Replace with your own GEE-enabled Google Cloud project ID
EE_PROJECT = "your-gee-project-id"
ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")


## 2. Upload representative stations (Final_Stations / Representative_Stations_Rainfall sheet with lat/lon)

In [ ]:
from google.colab import files
import pandas as pd

print("Select Final_Stations.xlsx (output from Tab 1)...")
uploaded = files.upload()
stations_path = list(uploaded.keys())[0]

stations = pd.read_excel(stations_path)
stations = stations[["Station_ID", "Latitude", "Longitude"]]
print(stations)


## 3. Available GCMs in NASA/GDDP-CMIP6

In [ ]:
NEX_GDDP_CMIP6_MODELS = [
    "ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CESM2", "CESM2-WACCM",
    "CMCC-CM2-SR5", "CMCC-ESM2", "CNRM-CM6-1", "CNRM-ESM2-1", "EC-Earth3",
    "EC-Earth3-Veg-LR", "FGOALS-g3", "GFDL-CM4", "GFDL-ESM4", "GISS-E2-1-G",
    "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "IITM-ESM", "INM-CM4-8", "INM-CM5-0",
    "IPSL-CM6A-LR", "KACE-1-0-G", "KIOST-ESM", "MIROC-ES2L", "MIROC6",
    "MPI-ESM1-2-HR", "MPI-ESM1-2-LR", "MRI-ESM2-0", "NESM3", "NorESM2-LM",
    "NorESM2-MM", "TaiESM1", "UKESM1-0-LL",
]
print(f"{len(NEX_GDDP_CMIP6_MODELS)} models available:")
for m in NEX_GDDP_CMIP6_MODELS:
    print(" -", m)


## 4. Parameters — pick models, scenario, year range

In [ ]:
# Pick any subset (or ALL — note: ALL 33 models x 31 years x 8 stations is a large run).
SELECTED_MODELS = ["MPI-ESM1-2-HR", "EC-Earth3", "MIROC6"]  # edit as needed

# Use SELECTED_MODELS = NEX_GDDP_CMIP6_MODELS to run every model

SCENARIO = "historical"   # historical covers 1950-2014, matches IMD historical period
START_YEAR = 1984
END_YEAR = 2014

CHUNK_YEARS = 5  # extract in N-year chunks to keep each GEE request small


## 5. Build station FeatureCollection

In [ ]:
station_features = []
for _, row in stations.iterrows():
    pt = ee.Geometry.Point([row["Longitude"], row["Latitude"]])
    station_features.append(ee.Feature(pt, {"Station_ID": row["Station_ID"]}))

station_fc = ee.FeatureCollection(station_features)
print(f"{station_fc.size().getInfo()} station points loaded into Earth Engine.")


## 6. Extraction function

For each model + year-chunk: filter NASA/GDDP-CMIP6 to that model/scenario/date range,
take the `pr` band, map `reduceRegions` (mean at each station point) over the
ImageCollection, flatten to a FeatureCollection, pull to a DataFrame via `getInfo()`.

Unit conversion: NEX-GDDP-CMIP6 `pr` is in **kg/m^2/s** (precipitation flux).
mm/day = pr * 86400.


In [ ]:
import time

PR_UNIT_TO_MM_DAY = 86400.0  # kg/m2/s -> mm/day


def extract_model_chunk(model, scenario, start_date, end_date, station_fc):
    coll = (ee.ImageCollection("NASA/GDDP-CMIP6")
            .filter(ee.Filter.eq("model", model))
            .filter(ee.Filter.eq("scenario", scenario))
            .filterDate(start_date, end_date)
            .select("pr"))

    def reduce_image(img):
        date_str = img.date().format("YYYY-MM-dd")
        reduced = img.reduceRegions(collection=station_fc, reducer=ee.Reducer.mean(), scale=27830)
        return reduced.map(lambda f: f.set("date", date_str))

    flat = coll.map(reduce_image).flatten()

    # Pull only the needed properties to keep payload small
    flat = flat.select(["date", "Station_ID", "mean"])
    return flat.getInfo()


def extract_model(model, scenario, start_year, end_year, station_fc, chunk_years=5):
    all_records = []
    year = start_year
    while year <= end_year:
        chunk_end = min(year + chunk_years - 1, end_year)
        start_date = f"{year}-01-01"
        end_date = f"{chunk_end}-12-31"
        print(f"  {model}: {start_date} to {end_date} ...")

        for attempt in range(3):
            try:
                result = extract_model_chunk(model, scenario, start_date, end_date, station_fc)
                break
            except Exception as e:
                print(f"    retry {attempt+1} after error: {e}")
                time.sleep(5)
        else:
            raise RuntimeError(f"Failed to extract {model} {start_date}-{end_date} after retries.")

        for feat in result["features"]:
            props = feat["properties"]
            all_records.append({
                "Date": props["date"],
                "Station_ID": props["Station_ID"],
                "pr_kg_m2_s": props.get("mean"),
            })
        year = chunk_end + 1

    df = pd.DataFrame(all_records)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Precip_mm"] = df["pr_kg_m2_s"] * PR_UNIT_TO_MM_DAY

    # pivot to Date x Station_ID, same format as Representative_Stations_Rainfall
    pivot = df.pivot(index="Date", columns="Station_ID", values="Precip_mm").reset_index()
    pivot = pivot.sort_values("Date").reset_index(drop=True)
    return pivot


## 7. Run extraction for each selected model

In [ ]:
model_outputs = {}

for model in SELECTED_MODELS:
    print(f"Extracting {model} ...")
    df_model = extract_model(model, SCENARIO, START_YEAR, END_YEAR, station_fc, CHUNK_YEARS)
    model_outputs[model] = df_model
    print(f"  -> {df_model.shape[0]} days x {df_model.shape[1]-1} stations\n")

print("Done.")


## 8. Save & download one Excel file per model

In [ ]:
for model, df_model in model_outputs.items():
    fname = f"CMIP6_{model}_{SCENARIO}_{START_YEAR}-{END_YEAR}.xlsx"
    path = f"/content/{fname}"
    df_model.to_excel(path, index=False)
    files.download(path)
    print(f"Saved {fname}: {df_model.shape}")
